# Day 2 — You Are the QA Engineer

**Module 6 · Agentic RAG Testing**

---

Day 1 built the agent. Today you test it.

Not "run a script and see if the output looks right." **Systematic testing** — the kind that gives WidgetCo's engineering team something they can ship against. You'll start from three real incidents their support team already logged, design a test case for each one from scratch, write an LLM judge that makes the verdict reproducible, and run the whole suite against the live agent.

By the end you will have built:

| What | Starting from |
|---|---|
| 3 test scenarios grounded in real incidents | Support tickets WidgetCo's team actually filed |
| A golden dataset entry + hard negative for each | The specific wrong answer the agent has been giving |
| An LLM judge that makes each verdict reproducible | A prompt that encodes what "correct" means for this failure mode |
| An extended coverage matrix | The same one from Module 4 Day 4, with today's failure modes added |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Cells that call the live agent need `.env` configured in this `examples/` folder (see Day 1's setup note). The golden-design and judge-definition cells run without credentials.

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both techniques come back — pointed at the planner's memory and its reasoning across hops instead of a single retrieval. And you're not reading about it. You're doing it.

---

## Your brief

**WidgetCo** is a B2B SaaS company founded in 2019. They build project-management and analytics tools for engineering teams — think sprint tracking, velocity dashboards, incident retrospectives. They're not a huge company. About 120 people, bootstrapped until a Series A in 2021, and still small enough that the Head of Support knows every enterprise account by name.

Their product line has evolved:

- **WidgetPro 2000** launched at founding. Solid product, but they sunset it in 2023 and replaced it with WidgetPro 3000. The 2000 had a `$50` cancellation fee; the 3000 dropped it entirely — cancel anytime, `$0`.
- **TurboMax** is their analytics add-on. It was called TurboMax 5 until a 2024 rebrand renamed it TurboMax Pro. Same `$299/year` price, new name.
- **Headquarters** moved from Austin to Denver in 2022 when they expanded the engineering team. The Denver office is lean — no walk-in support counter, everything handled online.

In 2023 they launched an AI support agent to handle the first line of customer queries — product questions, pricing, policies. By 2024 it was handling ~500 queries a day. Nobody tested it systematically when it shipped. It just worked well enough.

Then, three weeks ago, the Head of Support sent this:

> *"I need to flag something before our marketing campaign goes live. We've had three incidents from the AI agent that I can't explain. A customer migrating from WidgetPro 2000 was told the cancellation fee is $50 — it should be $0 on their new product. Another customer drove to the Denver office expecting to walk in and found a locked building. And a TurboMax Pro customer is disputing a $25 cancellation charge — I've checked every system, we never billed them $25, and that policy doesn't exist. We're about to 3× query volume. I need these fixed and tested before we flip the switch.*
>
> *Three incidents. Three different failure types. I can send you the exact queries if you need them."*

That last line — *three different failure types* — is the job. You're not here to fix the agent. You're here to write tests that catch these failures before the next release, on every future version of the agent.

Here's what you know going in:

| Incident | What the customer asked | What the agent said | What should have happened |
|---|---|---|---|
| 1 | Cancellation fee for the replacement for WidgetPro 2000 | `$50` | `$0` — that's WidgetPro 3000's fee |
| 2 | Same question, different run | `$0` (no product named) | `$0 for WidgetPro 3000` — product must be named |
| 3 | Cancellation fee for TurboMax Pro | `$25` | Admit the gap — no such policy exists |

By end of this notebook, each incident has a golden dataset entry, a hard negative that would have caught it, and an LLM judge that makes the verdict reproducible on every future agent version.

---

## Step 1 — Understand why your existing tools won't catch these incidents

Before writing a single test, you need to know which tools you already have and exactly where they break down. WidgetCo already has RAGAS from Module 5. You run it on the three incidents. Here's what happens:

**Incident 1 (wrong fee):** The agent's answer is *"The cancellation fee for the product that replaced WidgetPro 2000 is $50."* RAGAS `faithfulness` scores this **high** — `$50` is grounded in the retrieved context (it really was WidgetPro 2000's fee). `answer_relevancy` scores this **high** — the answer addresses the question. Both metrics pass. The customer is still wrong.

**Incident 2 (dropped identity):** The agent answers *"The cancellation fee is $0."* RAGAS scores this fine. The fee is correct, the answer is relevant. But which product? Nobody wrote a metric that checks what made it out of the retrieval loop into the final sentence.

**Incident 3 (fabricated fee):** The agent answers *"The cancellation fee for TurboMax Pro is $25."* RAGAS `faithfulness` flags this — `$25` doesn't appear in any retrieved chunk. But here's the problem: the agent's retrieval succeeded (it found the subscription price). If faithfulness sees *any* retrieved context, it may partially pass even on a fabricated answer, depending on how strictly the scorer is calibrated.

None of this is a flaw in RAGAS. These metrics were built for a system that retrieves exactly once:

- **`faithfulness`** asks "is every claim backed by some retrieved chunk?" — it cannot ask "was that the *right* chunk for this *entity*"
- **`answer_relevancy`** asks "does the response address the question?" — a confident wrong answer scores the same as a confident right one
- **`context_precision` / `context_recall`** score the *retrieval*, not whether the generator used it correctly

The failure modes in incidents 1–3 only become visible once you look at the *combination* of facts, not the individual facts. That's what you're building today: checks that look at what the agent did with what it retrieved.

> **The rule:** if an existing metric would catch an incident, you don't need a new one. If it wouldn't — even when you run it correctly — you do. All three incidents above need new checks.

---

## Step 2 — How to design a test case from an incident

Every test case in this notebook was designed using the same five-step process. Walk through it once before you see the first scenario — then you'll recognise it in each one.

**Step 1: Name the failure mode.**
Map the incident to one of the failure modes Day 1 defined. Is this premature_stop (the planner quit too early)? reasoning_chain_break (right facts, wrong combination or dropped identity)? ungraceful_failure (fabricated when it should have hedged)? Naming it before you write the test keeps you from writing a check that tests the wrong thing.

**Step 2: Write the question a real customer would ask.**
Not a synthetic benchmark question. The question that produced the incident. For incident 1, that's *"What is the cancellation fee for the product that replaced WidgetPro 2000?"* — because that's what a migrating customer actually asks.

**Step 3: Trace the hops the question requires.**
Map the question to the knowledge base facts it needs. How many retrievals does it take? Which fact comes first, which depends on which? This becomes your equivalence partition: is this a 1-hop question or a 2-hop question? A test suite with only 1-hop questions misses every multi-hop failure mode.

**Step 4: Write the golden entry.**
The golden is the *correct* answer and the conditions that define it. For reasoning-level checks, that means:
- `must_include`: facts the answer must mention (e.g. `["WidgetPro 3000", "$0"]`)
- `must_not_include`: facts that signal the wrong combination (e.g. `["$50"]`)
- `reference`: the ground-truth answer in plain English

**Step 5: Write the hard negative.**
The hard negative is the specific wrong answer you're testing for — the one that actually occurred in the incident. Not a random wrong answer. The *exact* failure mode. For incident 1, that's `"The cancellation fee for the product that replaced WidgetPro 2000 is $50."` — because that's what customers received.

**Step 6: Write the judge.**
The judge is an LLM prompt that takes the answer and returns `passed/score/reasoning`. The prompt must encode what "correct" means for this specific failure mode — not a generic "is this a good answer?" but "did the agent apply the RIGHT product's fee, not the discontinued one's?"

Every scenario below follows this sequence. Read the incident, watch the design decisions, then see the judge catch the hard negative.

---

## Scenario A — "The Memory Drop"
### Incident 2 · What happened, and what you're testing for

A WidgetPro 2000 customer was migrating their team to WidgetPro 3000. Before completing the move they wanted to confirm: is there a cancellation fee on the old product? The agent gave them the right number — `$0` — but never said which product it applied to.

The customer said thank you and went ahead with the migration.

Two weeks later they were back, disputing the answer. They had screenshot proof of what the agent told them. But the screenshot read *"The cancellation fee is $0."* They couldn't verify whether that `$0` was for WidgetPro 2000 (the one they were canceling) or WidgetPro 3000 (the one they were migrating to). They'd assumed it was their current product. They were right — but they had no way to know that from the answer.

This is a subtle failure. The number is correct. The agent retrieved the right information across two hops. It just dropped the product name when it wrote the final sentence. The fee made it through. The identity didn't.

**As the tester, your question is:** how do you write a test that catches *this specific omission* — not wrong numbers, not hallucinations, but a correct answer that's missing the one piece of context a customer needs to trust it?

The golden design cell below shows exactly how.

### Golden design: `multi-hop-widgetpro-01-memory-drop`

```
Question         "What is the cancellation fee for the product that replaced WidgetPro 2000?"

Hop trace        Hop 1  →  "WidgetPro 2000 was discontinued... replaced by WidgetPro 3000"  →  identity
                 Hop 2  →  "WidgetPro 3000's cancellation fee is $0"                         →  fee

Golden ref       "WidgetPro 3000's cancellation fee is $0."
must_include     ["WidgetPro 3000"]     ←  hop-1 product identity MUST survive into the final answer
must_not_include (none)                 ←  any correct fee that names the product is fine
eval_type        memory
```

**Why `must_include: ["WidgetPro 3000"]` and not `["$0"]`?**
The fee `$0` is what RAGAS would check — it's the headline fact. `WidgetPro 3000` is the hop-1 conclusion that can silently disappear when the generator assembles its final response. A customer reading `"The cancellation fee is $0"` cannot verify which product that applies to without already knowing the answer. The judge's job is to confirm the product name survived the hop boundary — not just that a dollar figure appeared.

**Hard negative (pre-scripted):**
```
response:  "The cancellation fee is $0."
verdict:   FAIL — must_include ["WidgetPro 3000"] absent
```
RAGAS faithfulness on this response: **~1.0** (`$0` is grounded). Our `memory_intact` judge: **FAIL**. That gap is what this test closes.

**Design decision:** This is a `must_include` check, not a `must_not_include` check. The fee itself isn't wrong — the omission is what's wrong. You're verifying that a specific entity crossed the hop boundary, not that a specific wrong value appeared. That distinction determines whether you write an inclusion rule or an exclusion rule.

In [31]:
import json
from pydantic import BaseModel
from agent import agentic_rag, judge_client, judge_model

# Load all golden cases once — keyed by id, reused across all 3 checks below.
with open("golden_dataset.json") as f:
    _golden = {c["id"]: c for c in json.load(f)}


In [32]:
class MemoryRetentionVerdict(BaseModel):
    retained: bool
    reasoning: str


async def judge_memory_retention(question: str, must_include: list[str], answer: str) -> MemoryRetentionVerdict:
    prompt = (
        f"Question: {question}\n\n"
        "A correct answer must explicitly reference all of the following facts from earlier retrieval hops:\n"
        + "\n".join(f"- {f}" for f in must_include)
        + f"\n\nAnswer to evaluate: {answer}\n\n"
        "Did the answer retain all required facts? A memory failure is when the answer gives correct "
        "information but drops the specific entity it belongs to — e.g. the right fee but no mention "
        "of which product it applies to. Respond with retained=true only if every required fact appears."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=MemoryRetentionVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Real agent answer — positive case, live model call.
memory_case = _golden["multi-hop-widgetpro-01-memory-drop"]
day1_result = await agentic_rag(memory_case["user_input"])
answer_with_memory = day1_result.response

# Hard negative from golden_dataset.json — hop 1's product identity dropped.
answer_memory_dropped = memory_case["response"]

for label, answer in [("REAL AGENT (memory intact)", answer_with_memory),
                       ("HARD NEGATIVE (memory dropped)", answer_memory_dropped)]:
    verdict = await judge_memory_retention(memory_case["user_input"], memory_case["must_include"], answer)
    print(f"[{label}] -> retained={verdict.retained}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print(f"day1_result.hit_max_hops = {day1_result.hit_max_hops}  (num_hops={day1_result.num_hops})")
print("False here means the planner confirmed it had enough — worth checking whenever")
print("a memory-validation case unexpectedly fails, since True would explain a lot on its own.")


[REAL AGENT (memory intact)] -> retained=True
    reasoning: The answer explicitly names both required facts: (1) it identifies that the product being asked about is the one that replaced WidgetPro 2000, and (2) it explicitly states the name of that replacement product, WidgetPro 3000, by including it in parentheses. The cancellation fee amount is given in relation to this named product.
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 (WidgetPro 3000) is $0.'
[HARD NEGATIVE (memory dropped)] -> retained=False
    reasoning: The answer states a cancellation fee but does not specify the product it applies to. The required fact from earlier hops is 'WidgetPro 3000,' which must be mentioned in the answer as the product associated with the cancellation fee. Since the answer only says 'The cancellation fee is $0' without naming the product, it fails to retain the required entity.
    answer: 'The cancellation fee is $0.'

day1_result.hit_max_hops = False  (num_

---

## Scenario B — "The Bait-and-Switch Fee"
### Incident 1 · What happened, and what you're testing for

A customer had just completed their migration from WidgetPro 2000 to WidgetPro 3000 and asked the agent about canceling. They got back: *"The cancellation fee for the product that replaced WidgetPro 2000 is $50."*

They nearly paid it.

WidgetPro 2000's cancellation fee really was `$50` — before it was discontinued. WidgetPro 3000, the replacement, dropped that fee entirely. Cancel anytime, no charge. But the agent pulled the old product's fee and attached it to the new product.

The support team flagged this as the worst of the three incidents because there's a paper trail. The customer has a screenshot of the agent quoting them `$50`. If they'd paid it based on that answer, WidgetCo would owe them a refund and an explanation.

What makes this failure particularly hard to catch: the agent didn't hallucinate. Both facts it retrieved are true. *WidgetPro 2000 was replaced by WidgetPro 3000* — true. *WidgetPro 2000's cancellation fee was $50* — true. The problem is what it did with those two facts: it took the fee from the old product and applied it to the new one.

Your existing RAGAS metrics from Module 5 won't catch this. Every individual claim is grounded. Answer relevancy is fine — the response is clearly about cancellation fees. The failure is in the combination, and no per-fact metric can see combinations.

**As the tester, your question is:** how do you write a test that specifically catches the wrong entity being cited — not "is the number grounded" but "is it the right product's number?"

The golden design cell below shows exactly how, and exactly why `must_not_include: ["$50"]` does something `faithfulness` cannot.

### Golden design: `multi-hop-widgetpro-01-hardneg`

```
Question         "What is the cancellation fee for the product that replaced WidgetPro 2000?"

Hop trace        Hop 1  →  "WidgetPro 2000 was discontinued... replaced by WidgetPro 3000"    →  identity
                 Hop 2  →  "WidgetPro 2000's cancellation fee was $50 before it was discontinued"
                            (wrong hop-2 — agent retrieved the OLD product's fee, not the new one's)

Golden ref       "WidgetPro 3000's cancellation fee is $0."
must_include     ["WidgetPro 3000", "$0"]    ←  both the replacement product AND its fee must appear
must_not_include ["$50"]                     ←  the discontinued product's fee must NOT appear as the answer
eval_type        reasoning
```

**Why both `must_include` AND `must_not_include`?**
`must_include` ensures the answer names the right entity and the right value. `must_not_include` is the sharper check: it specifically blocks the exact wrong value the agent produced in the real incident. Together they define a tight corridor — any answer inside that corridor is correct by construction.

**The hard negative in detail:**
```
retrieved[0]:  "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."
retrieved[1]:  "WidgetPro 2000's cancellation fee was $50 before it was discontinued."
response:      "The cancellation fee for the product that replaced WidgetPro 2000 is $50."
```
Both retrieved facts are individually grounded and true. The hard negative passes `faithfulness` (RAGAS). It fails `reasoning_chain_correct` (our judge). That gap — visible here, not in RAGAS — is the reason this test exists.

**Design decision:** `must_not_include: ["$50"]` is written for this *specific* failure mode, not as a generic exclusion. If you moved to a scenario where "$50" was the correct answer, you'd drop it. The value of the field is that it encodes what "wrong" looks like for *this* test case — not a universal policy.

**What the judge below is actually asking:**
> "The question asks about the REPLACEMENT product. Both facts are true individually. Did the agent apply the fee to the product it was asked about, or to the one that was discontinued?"

In [33]:
class ReasoningChainVerdict(BaseModel):
    correct_combination: bool
    reasoning: str


async def judge_reasoning_chain(
    question: str, must_include: list[str], must_not_include: list[str], answer: str
) -> ReasoningChainVerdict:
    prompt = (
        f"Question: {question}\n\n"
        f"A correct answer must include all of: {must_include}\n"
        f"A correct answer must NOT include any of: {must_not_include}\n\n"
        f"Answer to evaluate: {answer}\n\n"
        "Did the agent correctly combine the retrieved facts and reference the right entity? "
        "Both retrieved facts may be individually true — the failure is applying facts from the "
        "wrong entity (e.g. using WidgetPro 2000's fee to answer a question about its replacement). "
        "Respond with correct_combination=true only if must_include is satisfied AND must_not_include is absent."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=ReasoningChainVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Hard negative from golden_dataset.json — right facts, wrong combination.
reasoning_neg = _golden["multi-hop-widgetpro-01-hardneg"]

# Positive case — reuse the real agent answer from the memory check above (same question).
for label, answer in [("HARD NEGATIVE (should fail)", reasoning_neg["response"]),
                       ("REAL AGENT (should pass)", answer_with_memory)]:
    verdict = await judge_reasoning_chain(
        reasoning_neg["user_input"],
        reasoning_neg["must_include"],
        reasoning_neg["must_not_include"],
        answer,
    )
    print(f"[{label}] -> correct_combination={verdict.correct_combination}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print("A faithfulness check alone would PASS the hard negative above — '$50' really is in the")
print("retrieved context. The LLM judge catches the combination error because it reasons about")
print("which entity the question asked about, not just whether each fact is grounded.")


[HARD NEGATIVE (should fail)] -> correct_combination=False
    reasoning: The answer incorrectly states a cancellation fee of $50 for the replacement product. The must_include requirements ['WidgetPro 3000', '$0'] are not satisfied (WidgetPro 3000 is missing, and $0 is not stated). The must_not_include condition ['$50'] is violated, since $50 is present. The agent failed to correctly combine facts by applying the fee from the wrong entity (WidgetPro 2000's fee) instead of the replacement product's fee (WidgetPro 3000's $0 fee).
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $50.'
[REAL AGENT (should pass)] -> correct_combination=True
    reasoning: The answer includes 'WidgetPro 3000' and '$0', satisfying the must_include condition. It does not include '$50', satisfying the must_not_include condition. The answer correctly applies the cancellation fee of the replacement product (WidgetPro 3000) rather than the original product’s fee.
    answer: 'The c

---

## Scenario C — "The Phantom Policy"
### Incident 3 · What happened, and what you're testing for

A TurboMax Pro customer contacted support about a dispute. They said the AI agent had quoted them a `$25` cancellation fee, and now they were questioning a charge on their account.

The support team checked everything. WidgetCo has never had a `$25` cancellation fee for TurboMax Pro. The policy has never existed. There is no documentation of it anywhere. The agent made it up.

When you look at WidgetCo's knowledge base, you can see exactly why: they have TurboMax Pro's subscription price (`$299/year`), the product rename history (formerly TurboMax 5), and the general support plan details. They never wrote down a cancellation policy for TurboMax Pro. It just wasn't documented.

So the agent did what a poorly-constrained generator does when its retrieval loop finds nothing useful: it filled the gap. It generated a plausible-sounding number — `$25`, a figure that appears nowhere in the corpus — and presented it as fact.

This is the failure pattern Day 1 called out when covering the Klarna rollback. Simple queries matched humans; complex disputes didn't. The complex disputes were exactly this: cases where the agent needed to say "I don't have that information" and instead said something that sounded like an answer.

The correct behavior is a refusal to answer with a specific figure. Any number the agent produces here is wrong by definition — not because it's the wrong number, but because producing any number at all is the wrong response when the information doesn't exist.

**As the tester, your question is:** how do you write a test that checks whether an agent *admitted it didn't know* — when the honest answer is silence, not a number? A rule-based check can't enumerate every valid phrasing of "I don't know." An LLM judge can.

The golden design cell below shows how, and why this is the one scenario in today's notebook that can't be reduced to `must_include`/`must_not_include`.

### Golden design: `graceful-failure-01-hardneg`

```
Question         "What is the cancellation fee for TurboMax Pro?"

Hop trace        Hop 1  →  "TurboMax Pro's annual subscription costs $299"     (wrong fact — not a fee)
                 Hop 2  →  "TurboMax 5 was renamed to TurboMax Pro in 2024"    (also wrong fact)
                 Hop 3  →  same subscription price fact again
                 ⚠ hit_max_hops=True — planner NEVER confirmed enough_info

Golden ref       "This information is not available in the knowledge base."
must_include     (none — any admission of the gap is correct)
must_not_include (none in golden; any specific dollar figure is wrong)
eval_type        graceful
```

**Why there are no `must_include` or `must_not_include` fields for graceful failure:**
The correct answer is open-ended — any phrasing that admits the gap is valid. "I don't have that information," "that's not documented in our knowledge base," "I couldn't find a cancellation policy" — all correct. You can't enumerate the valid phrasings with a keyword list. This is why `ungraceful_failure` needs an LLM judge rather than a rule-based check.

**The hard negative in detail:**
```
retrieved contexts:  ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price."]
response:            "The cancellation fee for TurboMax Pro is $25."
```
`$25` appears nowhere in CORPUS. Not in any retrieved chunk, not in any product fact. This number was confabulated. A `faithfulness` check would flag it (no retrieved chunk contains `$25`) — but only if `faithfulness` has the right retrieved contexts. In a live run where the agent's retrieved contexts are logged separately from its answer, faithfulness might partially pass if the scorer sees context about TurboMax Pro even though it's about price, not fee.

**The `hit_max_hops` signal:**
Before running `judge_graceful_failure`, check `missing_fact_result.hit_max_hops`. If it's `True` (expected here), the loop exhausted hops without the planner confirming `enough_info` — meaning the generator was told to hedge and the judge is checking whether it did. If it's `False`, the planner hallucinated confidence on a question with no answer — that's its own finding worth logging separately.

**Design decision:** This golden entry has no `must_include` / `must_not_include` fields in the dataset intentionally. The judge encodes the criterion entirely in its prompt — *"graceful means explicitly admitting the gap; ungraceful means producing a specific fact not found in the knowledge base."* The field absence is a deliberate signal: this test family can't be reduced to keyword matching.

In [34]:
class GracefulFailureVerdict(BaseModel):
    is_graceful: bool
    reasoning: str


async def judge_graceful_failure(question: str, answer: str) -> GracefulFailureVerdict:
    prompt = (
        f"Question: {question}\n\n"
        "Context: The knowledge base contains no information that can answer this question.\n\n"
        f"Answer to evaluate: {answer}\n\n"
        "Did the agent handle the knowledge gap gracefully? "
        "Graceful means the agent explicitly admits it cannot find or does not have the answer — "
        "any phrasing that communicates 'the facts don't cover this' counts. "
        "Ungraceful means the agent invents or fabricates a specific fact (a number, date, policy) "
        "that was not in the knowledge base. Respond with is_graceful=true only for admissions of the gap."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=GracefulFailureVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Hard negative from golden_dataset.json — fabricated answer for a question with no corpus answer.
graceful_neg = _golden["graceful-failure-01-hardneg"]

# Real agent on the same question — corpus genuinely has no cancellation fee for TurboMax Pro.
missing_fact_result = await agentic_rag(graceful_neg["user_input"], verbose=True)
real_agent_answer = missing_fact_result.response

for label, answer in [("REAL AGENT", real_agent_answer),
                       ("HARD NEGATIVE (fabricated)", graceful_neg["response"])]:
    verdict = await judge_graceful_failure(graceful_neg["user_input"], answer)
    print(f"[{label}] -> is_graceful={verdict.is_graceful}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print(f"missing_fact_result.hit_max_hops = {missing_fact_result.hit_max_hops}")
print("This SHOULD be True — the corpus has no answer, so the planner should never confirm enough_info.")
print("If it's False, the planner hallucinated confidence — worth investigating separately.")
print()
print("If REAL AGENT came back is_graceful=False, that IS the finding this check exists to surface —")
print("it means the generator needs a stronger hedge instruction, not that the notebook is broken.")


[hop 1] query='What is the cancellation fee for TurboMax Pro?'
[hop 1] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge."]
[hop 1] planner.enough_info=False  reasoning="The user asked about the cancellation fee for 'TurboMax Pro', but the retrieved facts only mention its annual subscription price and the cancellation fee for a different product, 'WidgetPro 3000'. There is no information about TurboMax Pro's cancellation terms."
[hop 1] next_query='TurboMax Pro cancellation fee policy'

[hop 2] query='TurboMax Pro cancellation fee policy'
[hop 2] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", 'TurboMax 5 was renamed to TurboMax Pro in 2024 after a rebranding update.']
[hop 2] planner.enough_info=False  reasoning="The facts given cover TurboMax Pro's price and origin, but there is no direct information about 

---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones -- and they map directly onto this notebook's opening argument: `reasoning_chain_break` is the check for the failure faithfulness can't see, `ungraceful_failure` is the check for the failure no RAGAS metric asks about at all.


In [35]:
import json
from collections import Counter

with open("golden_dataset.json") as f:
    golden_cases = json.load(f)

categories    = sorted({c["category"] for c in golden_cases})
failure_modes = sorted({c["failure_mode"] for c in golden_cases})
counts        = Counter((c["category"], c["failure_mode"]) for c in golden_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

zero_cells = [(cat, fm) for cat in categories for fm in failure_modes if counts[(cat, fm)] == 0]
print()
if zero_cells:
    print("Still-empty cells (not necessarily a problem -- just visible now):")
    for cat, fm in zero_cells:
        print(f"- {cat} x {fm}")
else:
    print("No empty cells for these categories x failure modes -- Day 1's named premature_stop gap")
    print("is filled (see multi-hop-turbomax-01 in golden_dataset.json). single_hop_qa correctly")
    print("has no agentic-failure rows -- those columns only apply once a question needs multiple hops.")


                hallucination           premature_stop          reasoning_chain_break   ungraceful_failure      
multi_hop_qa    2                       2                       3                       2                       
single_hop_qa   2                       0                       0                       0                       

Still-empty cells (not necessarily a problem -- just visible now):
- single_hop_qa x premature_stop
- single_hop_qa x reasoning_chain_break
- single_hop_qa x ungraceful_failure


---
## Try It Yourself

1. Write a third memory-validation case where the final answer reflects hop 1's fact but **drops hop 2's entirely**. Does `facts_present_in_answer()` catch it the same way it caught the fully-dropped case above?
2. Add a `query_drift` row to `golden_dataset.json` -- a case where the reformulated query in hop 2 wanders away from the original question's intent (`query_drift` is named in Day 1's failure-mode table but has no row yet). What would its `retrieved_contexts` need to look like for a hard negative to actually demonstrate the drift, rather than just a wrong answer?
3. Write your own "right facts, wrong combination" hard negative in a domain other than product fees (e.g. dates, locations, prices) and add it to `golden_dataset.json` with `eval_type: "reasoning"`, `must_include`, and `must_not_include` fields (see `multi-hop-widgetpro-01` for the pattern). What made it easy or hard to construct compared to the WidgetPro example?

Exercise file: [`exercises/02_tool_memory_reasoning_exercise.md`](../exercises/02_tool_memory_reasoning_exercise.md)


---
## Summary — What you built as a QA engineer today

You started with three real support incidents. You ended with a reproducible test suite that would catch all three before the next release.

### What you built
| Scenario | Incident | Failure mode | Test mechanism |
|---|---|---|---|
| A — The Memory Drop | Customer got `$0` for an unnamed product | `reasoning_chain_break` (memory) | `must_include: ["WidgetPro 3000"]` + LLM judge |
| B — The Bait-and-Switch Fee | Customer was quoted `$50` on a `$0` product | `reasoning_chain_break` (wrong combination) | `must_include` + `must_not_include` + LLM judge |
| C — The Phantom Policy | Customer was quoted `$25` that doesn't exist | `ungraceful_failure` | `hit_max_hops` check + LLM judge |

### The thread back through the course
Every technique here was something you'd already seen — pointed one layer higher:

- **Equivalence partitioning (Module 4 Day 4)** → applied to hop count and failure modes, not just input ranges
- **Boundary value analysis (Module 5 Day 2)** → the hop boundary is the same kind of edge as a chunk boundary
- **Hard negatives (Module 4 Day 4 + Module 5 Day 3)** → right facts, wrong combination; grounded but fabricated
- **Coverage matrix (Module 4 Day 4)** → new columns for agentic failure modes, same habit
- **LangSmith tracing (Module 5)** → now load-bearing: without per-hop spans you can't tell Scenario A from Scenario B from a clean success in the logs

### What changed from Module 5
Module 5's RAGAS suite is still correct — and completely insufficient for these three incidents. Not because it's weak but because it was built for a different system. A faithfulness check on a single retrieval can't see a combination error across two hops. Answer relevancy can't see a memory drop. Neither can see what *should have* been retrieved and wasn't.

The right mental model: RAGAS tests the output. The three judges today test the *reasoning process* that produced it. Both layers are necessary — neither is enough alone.

### What WidgetCo gets
Before today: 500 queries a day, no systematic test coverage, three real incidents already logged.
After today: three targeted test cases, each with a hard negative that would have caught the real incident, run against a live agent, with reproducible LLM-judge verdicts that update automatically on every new model version.

**Next:** Module 7 — AI Agents Testing with DeepEval, where `judge_memory_retention`, `judge_reasoning_chain`, and `judge_graceful_failure` become formal, reusable metrics in a proper framework, and the agent gains real tool-calling instead of just retrieval.

---